# Sprint 1 - LLM Calls and Structured Outputs

This notebook introduces the shared helper core, the course OpenRouter model allowlist, direct chat calls, and structured JSON outputs with Pydantic plus LangGraph.

By the end, students should be able to identify the model boundary in the scaffold, call a model, and return validated data that another app component can trust.


## 1. Install the helper core from GitHub

Run this first in Colab. It clones or updates the course helper repository, then installs it editable so the flat modules in `src/` are importable.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/richhiey/ai-app-dev_Mod-A.git"
WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORK_DIR / "ai-app-dev_Mod-A"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
print(f"Installed helper core from {REPO_URL}")


## 2. Add your OpenRouter key

All model calls in this course go through OpenRouter. The helper rejects models outside the course allowlist before it makes a network request.


In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")


In [ ]:
from pydantic import BaseModel, Field

from models import ChatModel, all_allowed_model_ids
from openrouter import OpenRouterClient
from structured_graph import StructuredOutputGraph

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-1")
print("Course-enabled models:")
for model_id in all_allowed_model_ids():
    print("-", model_id)


## 3. Make a direct LLM call

Start with the simplest application boundary: a list of messages goes in, assistant text comes out.


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a concise AI app development teaching assistant.",
    },
    {
        "role": "user",
        "content": "In three bullets, explain why an app should call an LLM through a helper layer.",
    },
]

response = client.chat(
    messages,
    model=ChatModel.GEMINI_25_FLASH_LITE,
    temperature=0.2,
    max_tokens=300,
)
print(response.content)


## 4. Move from text to validated JSON

Structured output matters when the model result feeds routing, retrieval, UI state, or an automated action. The schema is the contract.


In [ ]:
class LessonRoute(BaseModel):
    task_type: str = Field(description="One of: chat, structured_output, retrieval, tools.")
    needs_retrieval: bool = Field(description="Whether the next step should search course material.")
    confidence: float = Field(ge=0, le=1, description="Confidence in the route.")
    next_action: str = Field(description="A short instruction for the app scaffold.")

route = client.structured(
    [
        {"role": "system", "content": "Route student requests for an AI app development notebook."},
        {"role": "user", "content": "I need examples of why naive RAG retrieves irrelevant chunks."},
    ],
    output_model=LessonRoute,
    schema_name="lesson_route",
    model=ChatModel.GEMINI_31_FLASH_LITE,
    temperature=0.1,
)

print(route.model_dump_json(indent=2))


## 5. Wrap the same idea in LangGraph

LangGraph makes the sequence explicit: prepare messages, call the model, return a validated object. Later notebooks use the same pattern for longer workflows.


In [ ]:
class LabPlan(BaseModel):
    objective: str
    steps: list[str] = Field(min_length=2, max_length=5)
    success_check: str

graph = StructuredOutputGraph(
    client=client,
    output_model=LabPlan,
    schema_name="lab_plan",
    system_prompt="Create compact campus lab plans for AI app development students.",
    model=ChatModel.GEMINI_31_FLASH_LITE,
)

plan = graph.invoke("Plan a 20 minute activity where students compare text output and JSON output.")
print(plan.model_dump_json(indent=2))


## Checkpoint

Before moving on, students should be able to point to the helper layer, the allowed model policy, the raw text path, and the structured output path.
